# About this notebook.

This notebook goes through all the texts and aplies on them a NLP pipeline consisting of (1) cleaning of the raw text, (2) sentence tokenization, (3) part-of-speech annotation, (4) lemmatization, and (5) named entity recognition.

The processed textual data are saved for future reuse.

In [3]:
import spacy
import os
import glob
from spacy.tokens import Doc
from spacy.language import Language
import pickle
from unidecode import unidecode
import sddk
import pandas as pd
import re
import sys
import importlib
import json
from spacy.tokens import Token
from spacy.language import Language

In [94]:
import pandas as pd

# Convert the sharing URL to a CSV export URL
sheet_id = "1bkHHTYc86K2IuEXqfYfkDNt5LovtvCU3gvqHIbVio88"
sheet_name = "Catalogue"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"

# Read directly into pandas
emlap_metadata = pd.read_csv(url, index_col=0).T
emlap_metadata.reset_index(inplace=True, names=["Author"])
emlap_metadata.head(5)

,Author,No.,is_done,is_noscemus,if_noscemus_id,"#if is_noscemus = True, don't transcribe",AUTHORSHIP,is_one_author,#if more than 1 author skip section and choose compendium below,is_author_known,...,genre,subject,SOURCE OF FILE,link,source_of_file,origin_of_copy,REFERENCES,catalogue_reference,secondary_references,general_comments
0,"Augurello, Chrysopoeia",100001,TRUE,TRUE,713324,NaN,NaN,TRUE,NaN,TRUE,...,Didactic Poem,alchemy,NaN,https://wiki.uibk.ac.at/noscemus/Chrysopoeia,Noscemus,Unknown,NaN,Noscemus Wiki,Soranzo 2019,The 1518 Basel version is also in Noscemus
1,"Pseudo-Lull, Secretis",100002,TRUE,FALSE,NaN,NaN,NaN,TRUE,NaN,TRUE,...,Treatise,NaN,NaN,https://www.digitale-sammlungen.de/en/view/bsb...,MDZ,MBS,NaN,Hirsch 1950,NaN,"There is a prior, 1514 edition of De secretis ..."
2,"Pantheus, Ars Transmutatione",100003,TRUE,FALSE,NaN,NaN,NaN,TRUE,NaN,TRUE,...,Treatise,NaN,NaN,NaN,GB,BL,NaN,NaN,NaN,This book was first published in 1518 with an ...
3,"Pantheus, Commentarium",100004,TRUE,FALSE,NaN,NaN,NaN,TRUE,NaN,TRUE,...,Treatise,alchemy,NaN,https://www.digitale-sammlungen.de/en/view/bsb...,MDZ,MSB,NaN,NaN,NaN,This 1519 book is catalogued wrongly by many l...
4,"Pantheus, Voarchadumia",100005,TRUE,FALSE,NaN,NaN,NaN,TRUE,NaN,TRUE,...,Treatise,NaN,NaN,NaN,ONB,ONB,NaN,NaN,NaN,Dedicated to Leonellus Marquis of Estense


In [95]:
emlap_automapping = pd.read_csv("../data/emlap_automapping.csv")
emlap_automapping["No."].tolist()

[100001.0,
 100002.0,
 100003.0,
 100004.0,
 100005.0,
 100006.0,
 nan,
 100008.0,
 100009.0,
 100010.0,
 100011.0,
 100012.0,
 100013.0,
 100014.0,
 100015.0,
 100016.0,
 100017.0,
 100018.0,
 100019.0,
 nan,
 100021.0,
 nan,
 100023.0,
 100024.0,
 100025.0,
 100026.0,
 100027.0,
 100028.0,
 100029.0,
 100030.0,
 nan,
 100032.0,
 100033.0,
 100034.0,
 nan,
 100036.0,
 100037.0,
 100038.0,
 100039.0,
 100040.0,
 nan,
 100042.0,
 100043.0,
 100044.0,
 100045.0,
 nan,
 100047.0,
 100048.0,
 100049.0,
 nan,
 nan,
 100052.0,
 100053.0,
 100054.0,
 100055.0,
 100056.0,
 nan,
 nan,
 100059.0,
 100060.0,
 100061.0,
 100062.0,
 100063.0,
 100064.0,
 nan,
 100066.0,
 nan,
 nan,
 100069.0,
 100070.0,
 nan,
 100072.0,
 100073.0,
 100074.0,
 100075.0]

In [96]:
emlap_automapping = emlap_automapping[emlap_automapping["No."].notnull()]
emlap_automapping["No."] = emlap_automapping["No."].astype(int).astype(str)

In [97]:
emlap_id_file_dict = dict(zip(emlap_automapping["No."], emlap_automapping["autofile"]))
emlap_id_file_dict

{'100001': nan,
 '100002': 'Pseudo-Lull1518_De_secretis_naturae_MDZ.json',
 '100003': 'Pantheus1518_Ars_Transmutationis_Metallicae_BL_GB.json',
 '100004': 'Pantheus1519_Commentarium_Transmutationis_Metallicae_MDZ.json',
 '100005': 'Pantheus1530_Voarchadumia_ONB.json',
 '100006': 'Savonarola1532_De_arte_conficiendi_aquam_vitae_ONB.json',
 '100008': 'Severinus1572_Epistola_MBZ_MBS.json',
 '100009': 'Vegius1518_Inter_inferiora_corpora_disputatio_ONB.json',
 '100010': 'Bracesco1548_De_alchemia_dialogi_II_IA_Madrid.json',
 '100011': 'De_alchemia1541_MDZ_MBS.json',
 '100012': 'Gessner1552_Thesaurus_Euonymi_Philiatri_ER_ZZ.json',
 '100013': 'Ulstad1525_Coelum_philosophorum_IA_BIUSP_pdf.json',
 '100014': 'Toxites1567_Spongia_stibii_MDZ_MBS.json',
 '100015': 'Gessner1569_Thesaurus_Euonymi_Philiatri_Liber_Secundus_MDZ_MBS.json',
 '100016': 'Bonus1546_Pretiosa_Margarita_Novella_ONB.json',
 '100017': 'Bodenstein1559_Isagoge_MDZ_MBS.json',
 '100018': nan,
 '100019': 'Ulstadt1526_De_epidemia_ONB.jso

In [99]:
emlap_metadata["filename"] = emlap_metadata["No."].apply(lambda x:  emlap_id_file_dict.get(x, None))
emlap_metadata.head(5)

,Author,No.,is_done,is_noscemus,if_noscemus_id,"#if is_noscemus = True, don't transcribe",AUTHORSHIP,is_one_author,#if more than 1 author skip section and choose compendium below,is_author_known,...,subject,SOURCE OF FILE,link,source_of_file,origin_of_copy,REFERENCES,catalogue_reference,secondary_references,general_comments,filename
0,"Augurello, Chrysopoeia",100001,TRUE,TRUE,713324,NaN,NaN,TRUE,NaN,TRUE,...,alchemy,NaN,https://wiki.uibk.ac.at/noscemus/Chrysopoeia,Noscemus,Unknown,NaN,Noscemus Wiki,Soranzo 2019,The 1518 Basel version is also in Noscemus,NaN
1,"Pseudo-Lull, Secretis",100002,TRUE,FALSE,NaN,NaN,NaN,TRUE,NaN,TRUE,...,NaN,NaN,https://www.digitale-sammlungen.de/en/view/bsb...,MDZ,MBS,NaN,Hirsch 1950,NaN,"There is a prior, 1514 edition of De secretis ...",Pseudo-Lull1518_De_secretis_naturae_MDZ.json
2,"Pantheus, Ars Transmutatione",100003,TRUE,FALSE,NaN,NaN,NaN,TRUE,NaN,TRUE,...,NaN,NaN,NaN,GB,BL,NaN,NaN,NaN,This book was first published in 1518 with an ...,Pantheus1518_Ars_Transmutationis_Metallicae_BL...
3,"Pantheus, Commentarium",100004,TRUE,FALSE,NaN,NaN,NaN,TRUE,NaN,TRUE,...,alchemy,NaN,https://www.digitale-sammlungen.de/en/view/bsb...,MDZ,MSB,NaN,NaN,NaN,This 1519 book is catalogued wrongly by many l...,Pantheus1519_Commentarium_Transmutationis_Meta...
4,"Pantheus, Voarchadumia",100005,TRUE,FALSE,NaN,NaN,NaN,TRUE,NaN,TRUE,...,NaN,NaN,NaN,ONB,ONB,NaN,NaN,NaN,Dedicated to Leonellus Marquis of Estense,Pantheus1530_Voarchadumia_ONB.json


For preprocessing the latin texts, we will use a module located outside of the current repository, specifically at the same level one level up.

The module can be clonned from here: https://github.com/CCS-ZCU/latin-preprocessing and imported to python following the steps below:

In [100]:
# for preprocessing the latin texts, we will use a module located outside of the current repository, specifically at the same level as the current project.
current_working_directory = os.getcwd()
relative_path = '../../latin-preprocessing/' # change according to your location...
module_path = os.path.abspath(os.path.join(current_working_directory, relative_path))
if module_path not in sys.path:
    sys.path.insert(0, module_path)
# Now import the module
import tomela

Tomela contains tuned latin preprocessing pipeline relying on spaCy and latinCy. You can check the pipeline as here:

In [101]:
tomela.nlp.pipeline

[('source_tracker', <function __main__.source_tracker(doc)>),
 ('senter', <spacy.pipeline.senter.SentenceRecognizer at 0x702cc483af30>),
 ('normer', <function la_core_web_lg.functions.normer(doc)>),
 ('tok2vec', <spacy.pipeline.tok2vec.Tok2Vec at 0x702cc483ac30>),
 ('tagger', <spacy.pipeline.tagger.Tagger at 0x702cc483ab10>),
 ('morphologizer',
  <spacy.pipeline.morphologizer.Morphologizer at 0x702cc483bd70>),
 ('trainable_lemmatizer',
  <spacy.pipeline.edit_tree_lemmatizer.EditTreeLemmatizer at 0x702d6ff3fdd0>),
 ('parser', <spacy.pipeline.dep_parser.DependencyParser at 0x702d6ffcdb60>),
 ('lookup_lemmatizer',
  <function la_core_web_lg.functions.make_lookup_lemmatizer_function(doc)>),
 ('ner', <spacy.pipeline.ner.EntityRecognizer at 0x702cc4c58b30>)]

In [114]:
tomela.nlp.max_length = 4000000

In [102]:
doc = tomela.nlp("Veritas, vt vlla dicit, semper est universalis et a principiis fundamentalis oritur (lib. 3, cap. VI)")
for token in doc:
    print((token.text, token.lemma_, token.pos_))

('Veritas', 'ueritas', 'NOUN')
(',', ',', 'PUNCT')
('vt', 'vt', 'ADV')
('vlla', 'vllus', 'NOUN')
('dicit', 'dico', 'VERB')
(',', ',', 'PUNCT')
('semper', 'semper', 'ADV')
('est', 'sum', 'AUX')
('universalis', 'uniuersalis', 'ADJ')
('et', 'et', 'CCONJ')
('a', 'ab', 'ADP')
('principiis', 'principium', 'NOUN')
('fundamentalis', 'fundamentalis', 'ADJ')
('oritur', 'orior', 'VERB')
('(', '(', 'PUNCT')
('lib', 'liber', 'NOUN')
('.', '.', 'PUNCT')
('3', '3', 'NUM')
(',', ',', 'PUNCT')
('cap', 'capitulum', 'NOUN')
('.', '.', 'PUNCT')
('VI', 'uis', 'NUM')
(')', ')', 'PUNCT')


In [103]:
source_path = "/srv/data/tome/tome-corpus/emlap_annotated_textblocks/"
len(os.listdir(source_path))

118

In [104]:
[f for f in sorted(os.listdir(source_path)) if "_params" not in f]

['Anon1550_De_alchemia_opuscula_MDZ_MBS.json',
 'Anon1550_Rosarium_philosophorum_Erara.json',
 'Auriferae_artisI1572_MBZ_Augsburg.json',
 'Barnaud1599_Quadriga_aurifera_IA_Madrid.json',
 'Bodenstein1559_Isagoge_MDZ_MBS.json',
 'Bonus1546_Pretiosa_Margarita_Novella_ONB.json',
 'Bracesco1548_De_alchemia_dialogi_II_IA_Madrid.json',
 'Claveus1598_Apologia_crysopoeiae_MDZ_MBS.json',
 'De_alchemia1541_MDZ_MBS.json',
 'Dorn1567_Clavis_totius_philosophiae_chymisticae_ONB_pdf.json',
 'Dorn1569_Artificii_chymistici_MDZ_MBS.json',
 'Dorn1570_Lapis_metaphysicus_MDZ_MBS_pdf.json',
 'Dorn1578_Theophrasti_Germani_Principis_MDZ_MBS.json',
 'Dorn1583_De_Naturae_luce_physica_MDZ_MBS.json',
 'DuChesne1575_Ad_Iacobi_Auberti_MDZ_Augsburg.json',
 'DuChesne1575_Sclopetarius_MDZ_Augsburg.json',
 'Erastus1578_Disputatio_de_auro_potabili_MDZ_MBS.json',
 'Fanianus1560_De_arte_metallicae_ONB_pdf.json',
 'Fanianus1576_De_arte_metallicae_MDZ_MBS.json',
 'Garlandius1560_Compendium_Alchemiae_SLUB_pdf.json',
 'Gessner

In [105]:
files_overview = []
for filename in os.listdir(source_path):
    #filename = 'DuChesne1575_Ad_Iacobi_Auberti_MDZ_Augsburg.json'
    if "_params" not in filename:
        filepath = os.path.join(source_path, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            textblocks = json.load(f)
        pages_n = len(textblocks)
        chars_n = sum([sum([len(tb["text"]) for tb in p]) for p in textblocks])
        files_overview.append({"filename" : filename, "pages_n" : pages_n, "chars_n" : chars_n})
pd.DataFrame(files_overview)

,filename,pages_n,chars_n
0,Moffett_De_iure_et_praestantia_MDZ_MBS.json,115,120620
1,Toxites1567_Spongia_stibii_MDZ_MBS.json,21,14543
2,Pseudo-Lull1567_Mercuriorum_liber_MDZ_MBS.json,405,340776
3,Pantheus1518_Ars_Transmutationis_Metallicae_BL...,53,49495
4,Pseudo-Aquinas1579_Secreta_alchemiae_magnalia_...,69,105063
5,Pseudo-Paracelsus1568_Pyrophilia_vexationumque...,153,136322
6,Bracesco1548_De_alchemia_dialogi_II_IA_Madrid....,131,246242
7,Hagecius1585_De_cervisia_ejusque_conficiendi_r...,65,73530
8,Pedemontanus1563_De_Secretis_MDZ_MBS.json,551,581004
9,Phaedro1562_Aquila_coelestis_MBZ_MBS.json,55,15562



# Develop and test with one example test

In [106]:
filename = 'DuChesne1575_Ad_Iacobi_Auberti_MDZ_Augsburg.json'
filepath = os.path.join(source_path, filename)
with open(filepath, 'r', encoding='utf-8') as f:
    textblocks = json.load(f)

In [107]:
len(textblocks)

93

In [108]:
textblocks[30][:10]

[{'coordinates': [165.1199951171875,
   40.31997299194336,
   427.239990234375,
   48.62395477294922],
  'text': '14\nRESPONSIO\n',
  'tag': 'header'},
 {'coordinates': [165.1199951171875,
   69.11996459960938,
   512.6112670898438,
   73.91996002197266],
  'text': 'calculo aut renum tartaro, tanta vi pro¬\n',
  'tag': 'text'},
 {'coordinates': [165.1199951171875,
   93.3840103149414,
   515.9405517578125,
   98.30400848388672],
  'text': 'desse diceres? Intelligo, Confugeres\n',
  'tag': 'text'},
 {'coordinates': [165.1199951171875,
   116.66397857666016,
   512.2813720703125,
   121.58397674560547],
  'text': 'ad sacram asinorum anchoram, nem¬\n',
  'tag': 'text'},
 {'coordinates': [168.72000122070312,
   140.87997436523438,
   509.6862487792969,
   145.6799774169922],
  'text': 'pe proprietatum occultarum: quod ta¬\n',
  'tag': 'text'},
 {'coordinates': [169.1999969482422,
   165.11996459960938,
   515.7952880859375,
   169.9199676513672],
  'text': 'men ipso sale fieri, qui illos r

In [109]:

# Modify the token extensions for simpler output
if not Token.has_extension("pages"):
    Token.set_extension("pages", default=None)
if not Token.has_extension("textblocks"):
    Token.set_extension("textblocks", default=None)
if not Doc.has_extension("char_to_source"):
    Doc.set_extension("char_to_source", default=None)


def process_textblocks(textblocks):
    full_text = ""
    char_to_source = {}

    for page_idx, page in enumerate(textblocks):
        for tb_idx, tb in enumerate(page):
            if tb["tag"] == "text":
                start_idx = len(full_text)
                text = tomela.text_cleaner(tb["text"])

                for char_idx in range(len(text)):
                    char_to_source[start_idx + char_idx] = {
                        "page_idx": page_idx,
                        "textblock_idx": tb_idx
                    }

                full_text += text

    return full_text, char_to_source


@Language.component("source_tracker")
def source_tracker(doc):
    if doc._.char_to_source is not None:
        for token in doc:
            # Get the character span of the entire token
            token_char_range = range(token.idx, token.idx + len(token.text))

            pages = set()
            textblocks = set()

            for char_idx in token_char_range:
                if char_idx in doc._.char_to_source:
                    source_info = doc._.char_to_source[char_idx]
                    pages.add(source_info["page_idx"])
                    textblocks.add(source_info["textblock_idx"])

            token._.pages = sorted(list(pages))
            token._.textblocks = sorted(list(textblocks))
    return doc


# Add the custom component to your existing pipeline if not already added
if "source_tracker" not in tomela.nlp.pipe_names:
    tomela.nlp.add_pipe("source_tracker", before="senter")


def process_with_source_tracking(textblocks, nlp):
    full_text, char_to_source = process_textblocks(textblocks)
    # Create the doc with the text
    doc = nlp.make_doc(full_text)
    # Set the char_to_source before running the pipeline
    doc._.char_to_source = char_to_source
    # Process the doc through each pipeline component
    for name, proc in nlp.pipeline:
        doc = proc(doc)
    return doc

In [110]:
textblocks[21:22]

[[{'coordinates': [146.63999938964844,
    46.55996322631836,
    435.6400146484375,
    51.599952697753906],
   'text': '5\nAD AVBERTVM.\n',
   'tag': 'header'},
  {'coordinates': [88.55999755859375,
    74.66397857666016,
    435.73681640625,
    79.58397674560547],
   'text': 'naturae iuuandum robur, & aduersus af¬\n',
   'tag': 'text'},
  {'coordinates': [91.19999694824219,
    98.66397857666016,
    444.1780700683594,
    103.58397674560547],
   'text': 'fectus melancholicos, ad exolutum ven¬\n',
   'tag': 'text'},
  {'coordinates': [88.80000305175781,
    122.87997436523438,
    438.64984130859375,
    127.67996978759766],
   'text': 'triculum; ad cardiacos, & praeter ratio¬\n',
   'tag': 'text'},
  {'coordinates': [90.72000122070312,
    146.87997436523438,
    441.6309814453125,
    151.6799774169922],
   'text': 'nem moestos efficax remedium. Certe\n',
   'tag': 'text'},
  {'coordinates': [89.27999877929688,
    172.31997680664062,
    441.3201904296875,
    177.11997985839844

In [111]:
doc = process_with_source_tracking(textblocks, tomela.nlp)


In [112]:
doc_sentdata = [(sent.text, [(t.text, t.lemma_, t.pos_, (t.idx - sent[0].idx, t.idx - sent[0].idx + len(t)), t._.pages, t._.textblocks) for t in sent]) for sent in doc.sents]
sent_data_updated = []
for n_sent, sent_data in enumerate(doc_sentdata):
    sent_data_updated.append((filename, n_sent, sent_data[0], sent_data[1]))

In [113]:
sent_data_updated[500:505]

[('DuChesne1575_Ad_Iacobi_Auberti_MDZ_Augsburg.json',
  500,
  'patrocinandonobis ipsis contradicere putet Aubertus) & grauis immixta terrae tenuissimaesulphuree, quae dicitur argentum uiuum,ex quo tanquam propinquiore materiamediante mixtione & actione sulphuris extrinseci, fit aurum, uel metallumaliud secundum maiorem aut minoremdigestionem ipsius naturae.',
  [('patrocinandonobis', 'patrocinandonobum', 'NOUN', (0, 17), [62], [8, 9]),
   ('ipsis', 'ipse', 'DET', (18, 23), [62], [9]),
   ('contradicere', 'contradico', 'VERB', (24, 36), [62], [9]),
   ('putet', 'puto', 'VERB', (37, 42), [62], [9]),
   ('Aubertus', 'Aubertus', 'PROPN', (43, 51), [62], [9, 10]),
   (')', ')', 'PUNCT', (51, 52), [62], [10]),
   ('&', '&', 'PUNCT', (53, 54), [62], [10]),
   ('grauis', 'grauis', 'ADJ', (55, 61), [62], [10]),
   ('immixta', 'immisceo', 'VERB', (62, 69), [62], [10]),
   ('terrae', 'terra', 'NOUN', (70, 76), [62], [10]),
   ('tenuissimaesulphuree',
    'tenuissimaesulphureus',
    'ADJ',
    (

In [43]:

# prepare function for working with segments...
def from_textblocks_to_doc(textblocks_pages):
    segment_len = 400
    if len(textblocks_pages) > segment_len: # if there is more than 400 pages
        segment_docs = []
        for n in range(0, len(textblocks_pages), segment_len):
            textblocks_segment = textblocks_pages[n:n+segment_len]
            segment_doc = process_with_source_tracking(textblocks_segment, tomela.nlp)

            segment_docs.append(segment_doc)
        doc = Doc.from_docs(segment_docs)
    else:
        doc = process_with_source_tracking(textblocks_pages, tomela.nlp)

    return doc


In [124]:
target_path = "/srv/data/tome/tome-corpus/sents_data_ids_jsons_v3-0/"
try:
    os.mkdir(target_path)
except:
    pass

In [125]:
emlap_metadata.head(5)

,Author,No.,is_done,is_noscemus,if_noscemus_id,"#if is_noscemus = True, don't transcribe",AUTHORSHIP,is_one_author,#if more than 1 author skip section and choose compendium below,is_author_known,...,subject,SOURCE OF FILE,link,source_of_file,origin_of_copy,REFERENCES,catalogue_reference,secondary_references,general_comments,filename
0,"Augurello, Chrysopoeia",100001,TRUE,TRUE,713324,NaN,NaN,TRUE,NaN,TRUE,...,alchemy,NaN,https://wiki.uibk.ac.at/noscemus/Chrysopoeia,Noscemus,Unknown,NaN,Noscemus Wiki,Soranzo 2019,The 1518 Basel version is also in Noscemus,NaN
1,"Pseudo-Lull, Secretis",100002,TRUE,FALSE,NaN,NaN,NaN,TRUE,NaN,TRUE,...,NaN,NaN,https://www.digitale-sammlungen.de/en/view/bsb...,MDZ,MBS,NaN,Hirsch 1950,NaN,"There is a prior, 1514 edition of De secretis ...",Pseudo-Lull1518_De_secretis_naturae_MDZ.json
2,"Pantheus, Ars Transmutatione",100003,TRUE,FALSE,NaN,NaN,NaN,TRUE,NaN,TRUE,...,NaN,NaN,NaN,GB,BL,NaN,NaN,NaN,This book was first published in 1518 with an ...,Pantheus1518_Ars_Transmutationis_Metallicae_BL...
3,"Pantheus, Commentarium",100004,TRUE,FALSE,NaN,NaN,NaN,TRUE,NaN,TRUE,...,alchemy,NaN,https://www.digitale-sammlungen.de/en/view/bsb...,MDZ,MSB,NaN,NaN,NaN,This 1519 book is catalogued wrongly by many l...,Pantheus1519_Commentarium_Transmutationis_Meta...
4,"Pantheus, Voarchadumia",100005,TRUE,FALSE,NaN,NaN,NaN,TRUE,NaN,TRUE,...,NaN,NaN,NaN,ONB,ONB,NaN,NaN,NaN,Dedicated to Leonellus Marquis of Estense,Pantheus1530_Voarchadumia_ONB.json


In [126]:
%%time
source_path = "/srv/data/tome/tome-corpus/emlap_annotated_textblocks/"
for id, filename in zip(emlap_metadata["No."], emlap_metadata["filename"]):
#for filename in .listdir(source_path):
    #if "_params" not in filename:
    try:
        if id + ".json" not in os.listdir(target_path):
            filepath = os.path.join(source_path, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                textblocks_pages = json.load(f)
            print("currently processing: ", filename)
            doc = process_with_source_tracking(textblocks_pages, tomela.nlp)
            doc_sentdata = [(sent.text, [(t.text, t.lemma_, t.pos_, (t.idx - sent[0].idx, t.idx - sent[0].idx + len(t)), t._.pages, t._.textblocks) for t in sent]) for sent in doc.sents]
            sent_data_updated = []
            for n_sent, sent_data in enumerate(doc_sentdata):
                sent_data_updated.append((id, n_sent, sent_data[0], sent_data[1]))
            with open(target_path + id + ".json", "w") as f:
                json.dump(sent_data_updated, f)
    except:
        print("failed with file: ", filename)
        pass

failed with file:  nan
currently processing:  Pseudo-Lull1518_De_secretis_naturae_MDZ.json
currently processing:  Pantheus1518_Ars_Transmutationis_Metallicae_BL_GB.json
currently processing:  Pantheus1519_Commentarium_Transmutationis_Metallicae_MDZ.json
currently processing:  Pantheus1530_Voarchadumia_ONB.json
currently processing:  Savonarola1532_De_arte_conficiendi_aquam_vitae_ONB.json
failed with file:  None
currently processing:  Severinus1572_Epistola_MBZ_MBS.json
currently processing:  Vegius1518_Inter_inferiora_corpora_disputatio_ONB.json
currently processing:  Bracesco1548_De_alchemia_dialogi_II_IA_Madrid.json
currently processing:  De_alchemia1541_MDZ_MBS.json
currently processing:  Gessner1552_Thesaurus_Euonymi_Philiatri_ER_ZZ.json
currently processing:  Ulstad1525_Coelum_philosophorum_IA_BIUSP_pdf.json
currently processing:  Toxites1567_Spongia_stibii_MDZ_MBS.json
currently processing:  Gessner1569_Thesaurus_Euonymi_Philiatri_Liber_Secundus_MDZ_MBS.json
currently processing:

In [140]:
target_path = "/srv/data/tome/tome-corpus/sents_data_jsons_v3-0/"


In [127]:
# Extract tahe lemmatized sentences

In [141]:
fns_jsons = os.listdir(target_path)
fns_jsons[:10]

['Moffett_De_iure_et_praestantia_MDZ_MBS.json',
 'Toxites1567_Spongia_stibii_MDZ_MBS.json',
 'Pseudo-Lull1567_Mercuriorum_liber_MDZ_MBS.json',
 'Pantheus1518_Ars_Transmutationis_Metallicae_BL_GB.json',
 'Pseudo-Aquinas1579_Secreta_alchemiae_magnalia_ONB.json',
 'Pseudo-Paracelsus1568_Pyrophilia_vexationumque_ONB.json',
 'Bracesco1548_De_alchemia_dialogi_II_IA_Madrid.json',
 'Hagecius1585_De_cervisia_ejusque_conficiendi_ratione.json',
 'Pedemontanus1563_De_Secretis_MDZ_MBS.json',
 'Phaedro1562_Aquila_coelestis_MBZ_MBS.json']

In [142]:
len(fns_jsons)

59

In [143]:
sents_data = json.load(open(target_path + fns_jsons[4], "r"))
sents_data[100:103]

[['Pseudo-Aquinas1579_Secreta_alchemiae_magnalia_ONB.json',
  100,
  'Similiter possimus dicere de aliis elementis,& ab istis elementis corpora supercoelestia esse composita peruirtutem diuinam aut intelligentiae regentis ipsam.',
  [['Similiter', 'similiter', 'ADV', [0, 9], [17], [15]],
   ['possimus', 'possum', 'VERB', [10, 18], [17], [15]],
   ['dicere', 'dico', 'VERB', [19, 25], [17], [15]],
   ['de', 'de', 'ADP', [26, 28], [17], [15]],
   ['aliis', 'alius', 'DET', [29, 34], [17], [15]],
   ['elementis', 'elementum', 'NOUN', [35, 44], [17], [15]],
   [',', ',', 'PUNCT', [44, 45], [17], [15]],
   ['&', '&', 'PUNCT', [45, 46], [17], [16]],
   ['ab', 'ab', 'ADP', [47, 49], [17], [16]],
   ['istis', 'iste', 'DET', [50, 55], [17], [16]],
   ['elementis', 'elementum', 'NOUN', [56, 65], [17], [16]],
   ['corpora', 'corpus', 'NOUN', [66, 73], [17], [16]],
   ['supercoelestia', '', 'VERB', [74, 88], [17], [16]],
   ['esse', 'sum', 'AUX', [89, 93], [17], [16]],
   ['composita', 'compositus',

In [145]:
lemmatized_sents_path = "/srv/data/tome/tome-corpus/lemmatized_sents_v3-0/"
try:
    os.mkdir(lemmatized_sents_path)
except:
    pass

In [146]:
for fn in fns_jsons:
    lemmatized_sents = []
    sents_data = json.load(open(target_path + fn, "rb"))
    for (doc_id, sent_id, sent_text, sent_data) in sents_data:
        lemmasent = []
        for wordform, lemma, tag, position, t_pages, t_textblocks in sent_data:
            if tag in ["NOUN", "PROPN", "ADJ", "VERB"]:
                lemmasent.append(lemma)
        lemmatized_sents.append(" ".join(lemmasent) + "\n")
    with open(lemmatized_sents_path + fn.replace(".json", ".txt"), "w", encoding="utf-8") as f:
        f.writelines(lemmatized_sents)